# Human Development Index

Written by Ines on 19th May 2026.

In [2]:
import rasterio
import numpy as np
import pandas as pd

def raster_to_csv(tif_path, output_path):
    """
    Extract lat, lon, and HDI value from a raster .tif file.
    """
    with rasterio.open(tif_path) as src:
        # Read the first band (HDI estimates)
        data = src.read(1)
        
        # Get the transform to convert pixel indices to coordinates
        transform = src.transform
        
        # Get row/col indices of all valid (non-NaN) pixels
        rows, cols = np.where(~np.isnan(data))
        
        # Convert pixel indices to lat/lon coordinates
        # rasterio gives us the center of each pixel
        lons, lats = rasterio.transform.xy(transform, rows, cols)
        
        # Get the corresponding HDI values
        values = data[rows, cols]
        
        # Build dataframe
        df = pd.DataFrame({
            'lat': lats,
            'lon': lons,
            'hdi': values
        })
        
        print(f"Extracted {len(df):,} valid grid cells")
        print(f"HDI range: {df['hdi'].min():.3f} - {df['hdi'].max():.3f}")
        print(df.head())
        
        df.to_csv(output_path, index=False)
        print(f"Saved to {output_path}")
    
    return df

# Usage
df_raster = raster_to_csv("hdi/hdi_raster_predictions.tif", "hdi_raster_output.csv")

Extracted 819,309 valid grid cells
HDI range: 0.041 - 1.000
     lat     lon       hdi
0  73.95   56.95  0.653000
1  73.95  110.05  0.689754
2  73.95  110.15  0.718193
3  73.95  110.25  0.708892
4  73.95  110.35  0.720881
Saved to hdi_raster_output.csv
